In [1]:
"""
Pathways construction from GTFS and OSM graph.
Generates walking connections between the starts/ends of routes that are near other routes' stops.
Output columns:
- start_route_id, end_route_id, start_stop_id, end_stop_id, walking_distance_m, walking_path_nodes
"""
import os
import math
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
from pathlib import Path

# Configure paths
GTFS_DIR = Path(r"C:/Users/ahmed/Downloads/gtfsAlex")
GRAPH_XML = Path(r"C:/Users/ahmed/Documents/grad/draft1/labeled.osm")

# Load GTFS
routes = pd.read_csv(GTFS_DIR / "routes.txt")
stops = pd.read_csv(GTFS_DIR / "stops.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")

# Ensure correct dtypes
stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce")

print(f"Loaded routes={len(routes)}, stops={len(stops)}, trips={len(trips)}, stop_times={len(stop_times)}")

# Load OSM graph (walkable)
g = ox.graph_from_xml(filepath=str(GRAPH_XML), bidirectional=True)
# Simplify graph for routing consistency
g = ox.convert.to_undirected(g)
print(f"Graph loaded with {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")


Loaded routes=104, stops=441, trips=192, stop_times=2547
Graph loaded with 45784 nodes, 65854 edges


In [2]:
# Build route -> ordered stops, and canonical start/end stop per route
# We assume stop_times stop_sequence ascending defines direction along a trip.

# Map trip -> route
trip_to_route = pd.Series(trips.route_id.values, index=trips.trip_id).to_dict()

# Choose a representative trip per route: the one with max number of stop_times
trip_counts = stop_times.groupby('trip_id').size().reset_index(name='n')
trip_counts['route_id'] = trip_counts['trip_id'].map(trip_to_route)
# Drop trips with unknown route_id
trip_counts = trip_counts.dropna(subset=['route_id'])
rep_trip_per_route = trip_counts.sort_values(['route_id','n'], ascending=[True, False]).drop_duplicates('route_id')

# Build ordered stops for each representative trip
ordered_stops = (
    stop_times[stop_times.trip_id.isin(rep_trip_per_route['trip_id'])]
    .sort_values(['trip_id','stop_sequence'])
    .merge(rep_trip_per_route[['trip_id','route_id']], on='trip_id', how='left')
)

# Pick start and end stop_id per route
start_stop_per_route = ordered_stops.groupby('route_id').first()['stop_id']
end_stop_per_route = ordered_stops.groupby('route_id').last()['stop_id']

print(f"Representative trips: {len(rep_trip_per_route)}, routes covered: {start_stop_per_route.index.nunique()}")

# Create convenience dicts
route_to_start_stop = start_stop_per_route.to_dict()
route_to_end_stop = end_stop_per_route.to_dict()

# Also collect all route->set(stops) for proximity to the whole route corridor
route_to_all_stops = (
    ordered_stops.groupby('route_id')['stop_id'].apply(lambda s: set(s.values)).to_dict()
)


Representative trips: 104, routes covered: 104


In [3]:
# Map each GTFS stop to nearest graph node
# Prepare stop_id -> (lat, lon) -> nearest node
stop_coords = stops.set_index('stop_id')[['stop_lat','stop_lon']]

# Vectorized nearest nodes using OSMnx
xs = stop_coords['stop_lon'].values
ys = stop_coords['stop_lat'].values
nearest_nodes = ox.distance.nearest_nodes(g, X=xs, Y=ys)

stop_to_node = {stop_id: node for stop_id, node in zip(stop_coords.index.values, nearest_nodes)}
print(f"Mapped {len(stop_to_node)} stops to nearest graph nodes")

# Convenience: node positions
node_x = nx.get_node_attributes(g, 'x')
node_y = nx.get_node_attributes(g, 'y')


Mapped 441 stops to nearest graph nodes


In [4]:
# Compute proximity candidates:
# For each route r1, find routes r2 whose start node is near any stop node of r1,
# and routes r3 whose end node is near any stop node of r1.
# We'll use a radius (meters) on great-circle distance to get candidates; exact walking
# shortest path will be computed in the next step.

from math import radians, sin, cos, asin, sqrt

def haversine_m(lat1, lon1, lat2, lon2):
    # returns meters
    R = 6371000.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

# Build stop_id -> (lat, lon)
stop_lat = stops.set_index('stop_id')['stop_lat'].to_dict()
stop_lon = stops.set_index('stop_id')['stop_lon'].to_dict()

# Precompute start/end node for every route
route_to_start_node = {r: stop_to_node[s] for r, s in route_to_start_stop.items() if s in stop_to_node}
route_to_end_node = {r: stop_to_node[s] for r, s in route_to_end_stop.items() if s in stop_to_node}

# Parameters
NEAR_RADIUS_M = 300  # coarse proximity filter in meters

# Build a fast lookup of route r -> list of nodes for all its stops
route_to_nodes = {r: [stop_to_node[s] for s in stops_set if s in stop_to_node]
                  for r, stops_set in route_to_all_stops.items()}

# For reverse lookup: node -> (lat, lon)
node_latlon = {n: (node_y.get(n), node_x.get(n)) for n in g.nodes()}

# Candidate list: (start_route_id, end_route_id, start_node, end_node)
# Two types:
# 1) r2 start near r1 corridor: (r2 -> r1)
# 2) r3 end near r1 corridor: (r3 -> r1)
# Note: We ignore r1's own start stop when considering proximity to other route starts.
proximity_candidates = []

route_ids = list(route_to_all_stops.keys())

for r1 in route_ids:
    r1_nodes = route_to_nodes.get(r1, [])
    if not r1_nodes:
        continue

    # compute r1's start node (to be excluded)
    r1_start_stop = route_to_start_stop.get(r1)
    r1_start_node = stop_to_node.get(r1_start_stop) if r1_start_stop in stop_to_node else None

    # 1) r2 starts near r1
    for r2, r2_start_node in route_to_start_node.items():
        if r2 == r1:
            continue
        # skip if the closest r1 node would be exactly its own start node
        lat2, lon2 = node_latlon.get(r2_start_node, (None, None))
        if lat2 is None:
            continue
        # coarse: check nearest r1 node by haversine radius
        near = False
        closest_r1_node = None
        closest_dist = float('inf')
        for n in r1_nodes:
            if r1_start_node is not None and n == r1_start_node:
                # ignore r1's start node when comparing to other starts
                continue
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat2, lon2)
            if d < closest_dist:
                closest_dist = d
                closest_r1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near and closest_r1_node is not None:
            proximity_candidates.append((r2, r1, r2_start_node, closest_r1_node))

    # 2) r3 ends near r1 (unchanged)
    for r3, r3_end_node in route_to_end_node.items():
        if r3 == r1:
            continue
        lat3, lon3 = node_latlon.get(r3_end_node, (None, None))
        if lat3 is None:
            continue
        near = False
        closest_r1_node = None
        closest_dist = float('inf')
        for n in r1_nodes:
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat3, lon3)
            if d < closest_dist:
                closest_dist = d
                closest_r1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near:
            proximity_candidates.append((r3, r1, r3_end_node, closest_r1_node))

print(f"Proximity candidates: {len(proximity_candidates)}")


Proximity candidates: 2129


In [5]:
# Shortest walking paths for candidates and assemble DataFrame
# We find the closest actual stop node on r1 to the candidate endpoint node as the "end_stop_id".
# Path cost uses edge length in meters if available, else fallback to 1.

# Build reverse node->stop_id index to resolve end_stop_id
node_to_stop_ids = {}
for sid, n in stop_to_node.items():
    node_to_stop_ids.setdefault(n, []).append(sid)

# Helper to pick the nearest r1 stop node to a given node by straight distance
from math import inf

def nearest_node_in_set(target_node, node_set):
    ty, tx = node_y.get(target_node), node_x.get(target_node)
    if ty is None:
        return None
    best_node, best_d = None, inf
    for n in node_set:
        ny, nx_ = node_y.get(n), node_x.get(n)
        if ny is None:
            continue
        d = haversine_m(ty, tx, ny, nx_)
        if d < best_d:
            best_d = d
            best_node = n
    return best_node

if not nx.get_edge_attributes(g, 'length'):
    g = ox.distance.add_edge_lengths(g)

records = []
failed = 0
for start_route_id, end_route_id, start_node, coarse_end_node in proximity_candidates:
    # refine end node: nearest actual route r1 stop node
    r1_nodes = route_to_nodes.get(end_route_id, [])
    if not r1_nodes:
        continue
    end_node = nearest_node_in_set(coarse_end_node, r1_nodes)
    if end_node is None:
        continue

    try:
        path = nx.shortest_path(g, source=start_node, target=end_node, weight='length')
        # compute distance
        dist = 0.0
        for u, v in zip(path[:-1], path[1:]):
            # get the first edge data (multi-edge safe)
            data = g.get_edge_data(u, v)
            if not data:
                continue
            # pick the first key
            ed = next(iter(data.values()))
            dist += float(ed.get('length', 1.0))

        # resolve stop ids
        # start_stop_id = the start stop of start_route_id
        start_stop_id = route_to_start_stop.get(start_route_id)
        
        # --- FIX STARTS HERE ---
        end_stop_id = None
        
        # 1. Get all GTFS stops mapped to this physical node
        candidates_at_node = node_to_stop_ids.get(end_node, [])
        
        # 2. Get the set of valid stops for the target route (end_route_id)
        #    (We computed 'route_to_all_stops' in Cell 20)
        valid_stops_for_route = route_to_all_stops.get(end_route_id, set())
        
        # 3. Find the intersection: The stop that is BOTH at this node AND on this route
        for candidate in candidates_at_node:
            if candidate in valid_stops_for_route:
                end_stop_id = candidate
                break
        
        # Fallback: If no strict match found (shouldn't happen if logic is perfect), 
        # take the first one or leave None
        if end_stop_id is None and candidates_at_node:
             end_stop_id = candidates_at_node[0]
        # --- FIX ENDS HERE ---

        records.append({
            'start_route_id': start_route_id,
            'end_route_id': end_route_id,
            'start_stop_id': start_stop_id,
            'end_stop_id': end_stop_id,
            'walking_distance_m': dist,
            'walking_path_nodes': path,
        })
    except Exception as e:
        failed += 1

print(f"Built {len(records)} pathways, failed={failed}")
pathways_df = pd.DataFrame.from_records(records)
# Optional: filter very short duplicates and self loops if any
pathways_df = pathways_df.drop_duplicates(subset=['start_route_id','end_route_id']).reset_index(drop=True)
pathways_df.head()


Built 2129 pathways, failed=0


,start_route_id,end_route_id,start_stop_id,end_stop_id,walking_distance_m,walking_path_nodes
0,RSdPdtwiGzlvedMSUpA36,-H9LP4vuOqj-RIRCZNg_6,322,321,139.949724,"[2712663257, 1605614943, 6953697314, 695369731..."
1,-Z1R0bmP-3IQrktLaw1-i,-H9LP4vuOqj-RIRCZNg_6,301,326,92.163069,"[4166052725, 1886821074, 1886821204]"
2,O-PLpyiKLXCmTOTN2ykl2,-H9LP4vuOqj-RIRCZNg_6,192,321,88.902979,"[2712663239, 5413673836, 2712663245]"
3,d1tk5YD606wPnGF4CLm4i,-H9LP4vuOqj-RIRCZNg_6,427,326,181.658840,"[6952133530, 1886821135, 11462532250, 1886821204]"
4,-H9LP4vuOqj-RIRCZNg_6,-Z1R0bmP-3IQrktLaw1-i,326,327,92.163069,"[1886821204, 1886821074, 4166052725]"


In [6]:
# Save to CSV and preview
output_csv = Path("pathways.csv")
pathways_df.to_csv(output_csv, index=False)
print(f"Saved pathways to {output_csv.resolve()}")
pathways_df.head(20)


Saved pathways to C:\Users\ahmed\Documents\grad\draft1\pathways.csv


,start_route_id,end_route_id,start_stop_id,end_stop_id,walking_distance_m,walking_path_nodes
0,RSdPdtwiGzlvedMSUpA36,-H9LP4vuOqj-RIRCZNg_6,322,321,139.949724,"[2712663257, 1605614943, 6953697314, 695369731..."
1,-Z1R0bmP-3IQrktLaw1-i,-H9LP4vuOqj-RIRCZNg_6,301,326,92.163069,"[4166052725, 1886821074, 1886821204]"
2,O-PLpyiKLXCmTOTN2ykl2,-H9LP4vuOqj-RIRCZNg_6,192,321,88.902979,"[2712663239, 5413673836, 2712663245]"
3,d1tk5YD606wPnGF4CLm4i,-H9LP4vuOqj-RIRCZNg_6,427,326,181.658840,"[6952133530, 1886821135, 11462532250, 1886821204]"
4,-H9LP4vuOqj-RIRCZNg_6,-Z1R0bmP-3IQrktLaw1-i,326,327,92.163069,"[1886821204, 1886821074, 4166052725]"
5,2HEouGQcgjyXwlsq69h3S,-Z1R0bmP-3IQrktLaw1-i,300,298,249.806522,"[9414993322, 7014712781, 7043828810, 941499332..."
6,50n7_gqFiIrgeNHtVzwF0,-Z1R0bmP-3IQrktLaw1-i,326,327,92.163069,"[1886821204, 1886821074, 4166052725]"
7,8CWoTtootKoeWNjoE5oOP,-Z1R0bmP-3IQrktLaw1-i,326,327,92.163069,"[1886821204, 1886821074, 4166052725]"
8,BKzwbCqAolUfyoXfpTwIC,-Z1R0bmP-3IQrktLaw1-i,325,327,1256.981165,"[4326474663, 11462717726, 5439510412, 43264746..."
9,cQIW-LxFXCIuxPpIw4MR2,-Z1R0bmP-3IQrktLaw1-i,296,296,0.000000,[5392019571]


In [7]:
# Visualization: plot two routes and the walking path between them
import folium
from itertools import pairwise

def _path_nodes_to_coords(G, node_path):
    return [(G.nodes[n]['y'], G.nodes[n]['x']) for n in node_path if 'x' in G.nodes[n] and 'y' in G.nodes[n]]

def _route_polyline_coords(G, route_id, ordered_stops_df, stop_to_node_map):
    seg_coords = []
    df = ordered_stops_df[ordered_stops_df['route_id'] == route_id].sort_values(['trip_id','stop_sequence'])
    # Use the first representative trip present for this route
    if df.empty:
        return []
    trip_id = df['trip_id'].iloc[0]
    sdf = df[df['trip_id'] == trip_id]
    stop_ids = sdf['stop_id'].tolist()
    node_seq = [stop_to_node_map[s] for s in stop_ids if s in stop_to_node_map]
    if len(node_seq) < 2:
        return []
    # Ensure edge lengths exist
    if not nx.get_edge_attributes(G, 'length'):
        ox.distance.add_edge_lengths(G)
    for u, v in pairwise(node_seq):
        try:
            sp = nx.shortest_path(G, u, v, weight='length')
            seg_coords.extend(_path_nodes_to_coords(G, sp))
        except Exception:
            # skip segment if routing fails
            continue
    return seg_coords

def plot_route_pair_with_path(start_route_id, end_route_id, save_html=None, map_tiles="cartodbpositron"):
    """
    Plot two routes (start_route_id, end_route_id) and the walking path between them
    using the precomputed record in pathways_df.

    Returns a Folium Map. If save_html is provided, saves the map to that path.
    """
    # Find pathway row
    row = pathways_df[(pathways_df['start_route_id'] == start_route_id) & (pathways_df['end_route_id'] == end_route_id)]
    if row.empty:
        raise ValueError("No pathway found for the given route pair in pathways_df")
    row = row.iloc[0]
    walking_nodes = row['walking_path_nodes']

    # Build map center around middle of walking path (fallback to first node)
    if walking_nodes and len(walking_nodes) > 0:
        mid_n = walking_nodes[len(walking_nodes)//2]
    else:
        # fallback: center on start route start stop
        mid_n = stop_to_node.get(route_to_start_stop.get(start_route_id))
    center_lat = g.nodes[mid_n]['y']
    center_lon = g.nodes[mid_n]['x']

    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles=map_tiles)

    # Route polylines
    r1_coords = _route_polyline_coords(g, start_route_id, ordered_stops, stop_to_node)
    r2_coords = _route_polyline_coords(g, end_route_id, ordered_stops, stop_to_node)

    if r1_coords:
        folium.PolyLine(r1_coords, color="#1f77b4", weight=5, opacity=0.8, tooltip=f"Route {start_route_id}").add_to(m)
    if r2_coords:
        folium.PolyLine(r2_coords, color="#ff7f0e", weight=5, opacity=0.8, tooltip=f"Route {end_route_id}").add_to(m)

    # Walking path
    walk_coords = _path_nodes_to_coords(g, walking_nodes)
    if walk_coords:
        folium.PolyLine(walk_coords, color="#2ca02c", weight=6, opacity=0.9, tooltip=f"Walk {len(walk_coords)} pts").add_to(m)
        # Mark start/end of walking path
        sy, sx = walk_coords[0]
        ey, ex = walk_coords[-1]
        folium.CircleMarker([sy, sx], radius=5, color="#2ca02c", fill=True, tooltip="Walk start").add_to(m)
        folium.CircleMarker([ey, ex], radius=5, color="#2ca02c", fill=True, tooltip="Walk end").add_to(m)

    # Mark start and end stops
    start_stop_id = route_to_start_stop.get(start_route_id)
    end_stop_id = route_to_end_stop.get(end_route_id)
    if start_stop_id in stop_to_node:
        ns = stop_to_node[start_stop_id]
        folium.Marker([g.nodes[ns]['y'], g.nodes[ns]['x']], icon=folium.Icon(color='blue', icon='play'), tooltip=f"Start route {start_route_id} start stop {start_stop_id}").add_to(m)
    if end_stop_id in stop_to_node:
        ne = stop_to_node[end_stop_id]
        folium.Marker([g.nodes[ne]['y'], g.nodes[ne]['x']], icon=folium.Icon(color='orange', icon='stop'), tooltip=f"End route {end_route_id} end stop {end_stop_id}").add_to(m)

    if save_html:
        m.save(save_html)
    return m

# Example usage (uncomment and pick actual ids present in pathways_df):
# plot_route_pair_with_path('r1_id', 'r2_id', save_html='pair_map.html')


In [8]:
pathways_df[pathways_df['start_route_id'] == '-H9LP4vuOqj-RIRCZNg_6']

,start_route_id,end_route_id,start_stop_id,end_stop_id,walking_distance_m,walking_path_nodes
4,-H9LP4vuOqj-RIRCZNg_6,-Z1R0bmP-3IQrktLaw1-i,326,327,92.163069,"[1886821204, 1886821074, 4166052725]"
209,-H9LP4vuOqj-RIRCZNg_6,50n7_gqFiIrgeNHtVzwF0,326,337,0.000000,[602781261]
351,-H9LP4vuOqj-RIRCZNg_6,8CWoTtootKoeWNjoE5oOP,326,337,0.000000,[602781261]
399,-H9LP4vuOqj-RIRCZNg_6,8NzGmmMDzhEhQEXbQjJTl,326,325,1164.818095,"[1886821204, 11462532250, 1886821135, 69521335..."
631,-H9LP4vuOqj-RIRCZNg_6,EqpMMSAPpyyeIs0OPJu7A,326,324,1159.659909,"[1886821204, 11462532250, 1886821135, 69521335..."
1236,-H9LP4vuOqj-RIRCZNg_6,d1tk5YD606wPnGF4CLm4i,326,323,181.658840,"[1886821204, 11462532250, 1886821135, 6952133530]"
1264,-H9LP4vuOqj-RIRCZNg_6,d591FuxnMulU7vr6uNxrB,326,338,57.316764,"[602781261, 5363987336, 5363987334, 602781295]"
1498,-H9LP4vuOqj-RIRCZNg_6,kdziSzfhBj4JiBeVoloxo,326,338,57.316764,"[602781261, 5363987336, 5363987334, 602781295]"


In [9]:

# Load GTFS routes file
routes_df = routes

# Create mapping from route_id to route_long_name (or route_short_name if you'd prefer)
route_id_to_name = routes_df.set_index('route_id')['route_long_name'].to_dict()

# Add start_route_name and end_route_name columns to pathways_df
pathways_df['start_route_name'] = pathways_df['start_route_id'].map(route_id_to_name)
pathways_df['end_route_name'] = pathways_df['end_route_id'].map(route_id_to_name)

# Show the result for all rows where start_route_id is the one in question
pathways_df[pathways_df['start_route_id'] == '-H9LP4vuOqj-RIRCZNg_6'][['start_route_id', 'start_route_name', 'end_route_id', 'end_route_name', 'start_stop_id', 'end_stop_id', 'walking_distance_m', 'walking_path_nodes']]


,start_route_id,start_route_name,end_route_id,end_route_name,start_stop_id,end_stop_id,walking_distance_m,walking_path_nodes
4,-H9LP4vuOqj-RIRCZNg_6,Ezbet Saad - Sidi Gabir,-Z1R0bmP-3IQrktLaw1-i,El-Awayed - Sidi Gabir,326,327,92.163069,"[1886821204, 1886821074, 4166052725]"
209,-H9LP4vuOqj-RIRCZNg_6,Ezbet Saad - Sidi Gabir,50n7_gqFiIrgeNHtVzwF0,Kilo 21 - Sidi Gabir,326,337,0.000000,[602781261]
351,-H9LP4vuOqj-RIRCZNg_6,Ezbet Saad - Sidi Gabir,8CWoTtootKoeWNjoE5oOP,Hanuvil - Sidi Gabir,326,337,0.000000,[602781261]
399,-H9LP4vuOqj-RIRCZNg_6,Ezbet Saad - Sidi Gabir,8NzGmmMDzhEhQEXbQjJTl,Sidi Bishr - Train Station (El-Shohada Square),326,325,1164.818095,"[1886821204, 11462532250, 1886821135, 69521335..."
631,-H9LP4vuOqj-RIRCZNg_6,Ezbet Saad - Sidi Gabir,EqpMMSAPpyyeIs0OPJu7A,Asafra - Train Station (El-Shohada Square),326,324,1159.659909,"[1886821204, 11462532250, 1886821135, 69521335..."
1236,-H9LP4vuOqj-RIRCZNg_6,Ezbet Saad - Sidi Gabir,d1tk5YD606wPnGF4CLm4i,El-Mawqaf El-Geded - Sidi Gabir,326,323,181.658840,"[1886821204, 11462532250, 1886821135, 6952133530]"
1264,-H9LP4vuOqj-RIRCZNg_6,Ezbet Saad - Sidi Gabir,d591FuxnMulU7vr6uNxrB,Asafra - Hadra,326,338,57.316764,"[602781261, 5363987336, 5363987334, 602781295]"
1498,-H9LP4vuOqj-RIRCZNg_6,Ezbet Saad - Sidi Gabir,kdziSzfhBj4JiBeVoloxo,El-Mansheya - Green Plaza Mall,326,338,57.316764,"[602781261, 5363987336, 5363987334, 602781295]"


In [10]:
# plot the 2 rotues and the walking path between them in pathways given 2 routes id
plot_route_pair_with_path('-H9LP4vuOqj-RIRCZNg_6', '-Z1R0bmP-3IQrktLaw1-i')